# Videovergleich für Ablauf2.1

Dieses Notebook verwendet den Trace `Ablauf2.1 08072026` aus `event_log.xes`. Die Liste der sichtbaren Events ist bewusst manuell gepflegt: Sie enthält nur Vorgänge, die in einer Fabrikaufnahme anhand von Bewegung, Stillstand, Ankunft oder sichtbarer Handhabung plausibel markiert werden können.

Die Zeitbasis des Logs bleibt UTC. Videozeitpunkte werden später als Sekunden seit Videobeginn eingetragen.

In [4]:
from __future__ import annotations

import json
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

XES_PATH = Path("event_log.xes")
TRACE_NAME = "Ablauf2.1 08072026"
XES_NAMESPACE = {"xes": "http://www.xes-standard.org/"}


def read_xes_trace(path: Path, trace_name: str) -> list[dict[str, str]]:
    root = ET.parse(path).getroot()
    for trace in root.findall("xes:trace", XES_NAMESPACE):
        trace_attributes = {
            child.attrib["key"]: child.attrib.get("value", "")
            for child in trace.findall("xes:string", XES_NAMESPACE)
        }
        if trace_attributes.get("concept:name") != trace_name:
            continue

        events = []
        for event in trace.findall("xes:event", XES_NAMESPACE):
            attributes = {}
            for child in event:
                key = child.attrib.get("key")
                if key:
                    attributes[key] = child.attrib.get("value", "")
            events.append(attributes)
        return events

    raise ValueError(f"Trace nicht gefunden: {trace_name}")


raw_events = read_xes_trace(XES_PATH, TRACE_NAME)
events = pd.DataFrame(raw_events)
events["timestamp"] = pd.to_datetime(events["time:timestamp"], utc=True)
events = events.sort_values("timestamp").reset_index(drop=True)

# Der XES-Export enthält je nach Quelle JSON- und JSONL-Wiederholungen.
events = events[events["source_file"].str.endswith(".jsonl", na=False)].copy()
events["details_dict"] = events["details"].apply(
    lambda value: json.loads(value) if isinstance(value, str) and value else {}
)
events["relative_s"] = (
    events["timestamp"] - events["timestamp"].min()
).dt.total_seconds()

event_counts = events["concept:name"].value_counts()
print(f"{len(events):,} JSONL-Events im Trace {TRACE_NAME!r}")
print(f"Zeitraum: {events['timestamp'].min()} bis {events['timestamp'].max()}")
print("\nHäufigste Eventtypen:")
print(event_counts.head(20).to_string())

2,542 JSONL-Events im Trace 'Ablauf2.1 08072026'
Zeitraum: 2026-07-08 08:02:36.500000+00:00 bis 2026-07-08 08:33:33.463000+00:00

Häufigste Eventtypen:
concept:name
ORDER_RECEIVED                             757
ORDER_STARTED                              757
ORDER_STOPPED                              171
FTS_clearLoadHandler                        64
MODULE_ORDER_DROP                           41
MODULE_DROP_FINISHED                        41
MODULE_ORDER_PICK                           34
MODULE_PICK_FINISHED                        34
MODULE_DROP_RUNNING                         32
MODULE_PICK_RUNNING                         32
MODULE_CHECK_QUALITY_RUNNING                27
ERROR_UNKNOWN                               24
MODULE_RGB_NFC_FINISHED                     18
ORDER_REQUEST                               18
ORDER_RESPONSE                              18
FTS_ORDER_SENT_TO_SVR4H73901                18
FTS_LOAD_HANDLING_STARTED_AT_SVR4H73901     18
FTS_LOAD_HANDLING_STOPPED_AT_SVR4H73

## Manuelle Auswahl visuell erkennbarer Events

Jedes ausgewählte Log-Event besitzt genau **einen Timestamp**. Eine Fahrt ist die einzige Ausnahme auf Prozessebene: Sie besteht aus zwei einzelnen Events, `FTS_DRIVING_STARTED_FROM_*` (Start) und `FTS_DRIVING_STOPPED_AT_*` (Ende). Die Fahrtdauer wird erst beim Vergleich aus diesen beiden Zeitpunkten berechnet.

| Eventfamilie | Log-Event bzw. Zeitpunkt | Im Video erkennbar? | Verwendung |
|---|---|---:|---|
| FTS fährt los | `FTS_DRIVING_STARTED_FROM_*` | Ja | Einzelner Startzeitpunkt einer Fahrt |
| FTS hält an | `FTS_DRIVING_STOPPED_AT_*` | Ja | Einzelner Endzeitpunkt derselben Fahrt |
| Beginn/Ende Handhabung | `FTS_LOAD_HANDLING_STARTED_AT_*`, `FTS_LOAD_HANDLING_STOPPED_AT_*` | Ja, mit Einschränkung | Jeweils einzelne sichtbare Zustandsmarke, keine künstliche Dauer |
| Einlagerung im HBW | Positiver `loadTimestamp` aus `HBW state.loads[]` | Ja | Einlagerungszeitpunkt am Lagerplatz; `0` bedeutet leer, wiederholte Zustände werden dedupliziert |
| PICK | `MODULE_PICK_FINISHED` | Ja | Einzelner Zeitpunkt des abgeschlossenen PICKs |
| DROP | `MODULE_DROP_FINISHED` | Ja | Einzelner Zeitpunkt des abgeschlossenen DROPs |
| Qualitätsprüfung | `MODULE_CHECK_QUALITY_*` | Eher nein | Interner Zustand, ohne sichtbares Ergebnis nicht sicher markierbar |
| Auftrag/MQTT | `ORDER_*`, `FTS_ORDER_SENT_*` | Nein | Interne Kommunikations- oder Zustandsereignisse |

Identische Export-Duplikate mit gleichem Eventnamen, Timestamp und Ziel werden vor der Auswertung entfernt.

In [5]:
VISIBLE_EVENT_DEFINITIONS = {
    "fts_driving_start": {
        "pattern": r"^FTS_DRIVING_STARTED_FROM_",
        "label": "FTS fährt los",
        "kind": "driving_start",
    },
    "fts_driving_end": {
        "pattern": r"^FTS_DRIVING_STOPPED_AT_",
        "label": "FTS hält an",
        "kind": "driving_end",
    },
    "load_handling_start": {
        "pattern": r"^FTS_LOAD_HANDLING_STARTED_AT_",
        "label": "Beginn der FTS-Handhabung",
        "kind": "point",
    },
    "load_handling_end": {
        "pattern": r"^FTS_LOAD_HANDLING_STOPPED_AT_",
        "label": "Ende der FTS-Handhabung",
        "kind": "point",
    },
    "pick": {
        "pattern": r"^MODULE_PICK_FINISHED$",
        "label": "PICK abgeschlossen",
        "kind": "point",
    },
    "drop": {
        "pattern": r"^MODULE_DROP_FINISHED$",
        "label": "DROP abgeschlossen",
        "kind": "point",
    },
}

HBW_JSONL_PATH = Path("../FT-data-20260612-original/Ablauf2.1 08072026.jsonl")
HBW_SERIAL_NUMBER = "SVR4H73901"


def hbw_load_event_rows(event_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    with HBW_JSONL_PATH.open(encoding="utf-8") as source:
        for line in source:
            record = json.loads(line)
            if record.get("topic") != f"module/v1/ff/{HBW_SERIAL_NUMBER}/state":
                continue

            for load in record.get("payload", {}).get("loads", []):
                load_timestamp = load.get("loadTimestamp")
                if not load_timestamp:
                    continue

                timestamp = pd.to_datetime(load_timestamp, unit="ms", utc=True)
                rows.append(
                    {
                        "family": "hbw_load",
                        "label": "Einlagerung im HBW",
                        "kind": "point",
                        "event_name": "HBW_LOAD_TIMESTAMP",
                        "log_timestamp": timestamp,
                        "log_time_s": (
                            timestamp - event_frame["timestamp"].min()
                        ).total_seconds(),
                        "location": load.get("loadPosition", ""),
                        "destination": HBW_SERIAL_NUMBER,
                        "load_type": load.get("loadType", ""),
                        "load_id": load.get("loadId", ""),
                    }
                )

    return pd.DataFrame(rows).drop_duplicates(
        subset=["log_timestamp", "location", "load_id"]
    )


def visible_event_rows(event_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for family, definition in VISIBLE_EVENT_DEFINITIONS.items():
        matching_events = event_frame[
            event_frame["concept:name"].str.match(definition["pattern"], na=False)
        ].sort_values("timestamp")

        for _, event in matching_events.drop_duplicates(
            subset=["concept:name", "timestamp", "location", "destination_node"]
        ).iterrows():
            rows.append(
                {
                    "family": family,
                    "label": definition["label"],
                    "kind": definition["kind"],
                    "event_name": event["concept:name"],
                    "log_timestamp": event["timestamp"],
                    "log_time_s": event["relative_s"],
                    "location": event.get("location", ""),
                    "destination": event.get("destination_node", ""),
                    "load_type": "",
                    "load_id": "",
                }
            )

    visible = pd.DataFrame(rows)
    visible = pd.concat(
        [visible, hbw_load_event_rows(event_frame)], ignore_index=True
    ).sort_values("log_timestamp").reset_index(drop=True)
    visible["occurrence"] = visible.groupby("family").cumcount() + 1
    return visible


visible_events = visible_event_rows(events)

output_path = Path("visible_events.csv")
visible_events.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"{len(visible_events):,} sichtbare Einzelereignisse gefunden")
print(f"Tabelle gespeichert unter: {output_path.resolve()}")

visible_events[[
    "family", "kind", "occurrence", "event_name", "log_timestamp", "log_time_s",
    "location", "destination", "load_type", "load_id",
]]

321 sichtbare Einzelereignisse gefunden
Tabelle gespeichert unter: C:\Users\olive\Documents\Uni\Forschungsprojekt\Zeitdatenanlyse\visible_events.csv


,family,kind,occurrence,event_name,log_timestamp,log_time_s,location,destination,load_type,load_id
0,fts_driving_start,driving_start,1,FTS_DRIVING_STARTED_FROM_SVR4H92774_TO_SVR4H73901,2026-07-08 08:03:11.096000+00:00,34.596,SVR4H92774,SVR4H73901,,
1,load_handling_start,point,1,FTS_LOAD_HANDLING_STARTED_AT_SVR4H92774,2026-07-08 08:03:12.504000+00:00,36.004,SVR4H92774,SVR4H73901,,
2,fts_driving_end,driving_end,1,FTS_DRIVING_STOPPED_AT_SVR4H92774_FROM_SVR4H73901,2026-07-08 08:03:12.508000+00:00,36.008,SVR4H92774,SVR4H73901,,
3,drop,point,1,MODULE_DROP_FINISHED,2026-07-08 08:03:26.237000+00:00,49.737,NaN,NaN,,
4,load_handling_end,point,1,FTS_LOAD_HANDLING_STOPPED_AT_SVR4H92774,2026-07-08 08:03:27.338000+00:00,50.838,SVR4H92774,SVR4H73901,,
...,...,...,...,...,...,...,...,...,...,...
316,fts_driving_start,driving_start,62,FTS_DRIVING_STARTED_FROM_SVR4H77011_TO_SVR4H92774,2026-07-08 08:32:54.690000+00:00,1818.190,SVR4H77011,SVR4H92774,,
317,load_handling_start,point,62,FTS_LOAD_HANDLING_STARTED_AT_SVR4H92774,2026-07-08 08:33:04.737000+00:00,1828.237,SVR4H92774,SVR4H92774,,
318,fts_driving_end,driving_end,62,FTS_DRIVING_STOPPED_AT_SVR4H92774_FROM_SVR4H92774,2026-07-08 08:33:04.748000+00:00,1828.248,SVR4H92774,SVR4H92774,,
319,pick,point,32,MODULE_PICK_FINISHED,2026-07-08 08:33:32.253000+00:00,1855.753,NaN,NaN,,


## Videozeitpunkte eintragen

Jede Video-Beobachtung erhält genau einen `video_timestamp_s` in Sekunden seit Beginn des Videos. Für eine Fahrt werden zwei Zeilen eingetragen: eine für `fts_driving_start` und eine für `fts_driving_end` mit derselben `occurrence`. Bei DROP, PICK und Handhabung wird nur ein einzelner Zeitpunkt eingetragen.

In [6]:
video_observations = pd.DataFrame(
    columns=["family", "occurrence", "video_timestamp_s", "note"]
)

# Beispiel: zwei einzelne Zeitpunkte für eine Fahrt.
# video_observations.loc[len(video_observations)] = {
#     "family": "fts_driving_start",
#     "occurrence": 1,
#     "video_timestamp_s": 125.4,
#     "note": "FTS setzt sich sichtbar in Bewegung",
# }
# video_observations.loc[len(video_observations)] = {
#     "family": "fts_driving_end",
#     "occurrence": 1,
#     "video_timestamp_s": 137.1,
#     "note": "FTS steht am Zielpunkt",
# }

# Beispiel: DROP ist ein Einzelereignis, keine Dauer.
# video_observations.loc[len(video_observations)] = {
#     "family": "drop",
#     "occurrence": 1,
#     "video_timestamp_s": 149.0,
#     "note": "Ablage abgeschlossen",
# }

video_observations

,family,occurrence,video_timestamp_s,note


In [7]:
def compare_with_video(
    log_events: pd.DataFrame,
    observations: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    empty_comparison = pd.DataFrame(
        columns=[
            "family", "occurrence", "label", "kind", "log_timestamp", "log_time_s",
            "video_timestamp_s", "timestamp_difference_s", "note",
        ]
    )
    empty_driving = pd.DataFrame(
        columns=[
            "occurrence", "log_driving_start_s", "log_driving_end_s",
            "video_driving_start_s", "video_driving_end_s", "log_driving_duration_s",
            "video_driving_duration_s", "driving_duration_difference_s",
        ]
    )
    if observations.empty:
        return empty_comparison, empty_driving

    comparison = observations.merge(
        log_events,
        on=["family", "occurrence"],
        how="left",
        validate="one_to_one",
    )
    comparison["timestamp_difference_s"] = (
        comparison["video_timestamp_s"] - comparison["log_time_s"]
    )

    result = comparison[
        [
            "family", "occurrence", "label", "kind", "log_timestamp", "log_time_s",
            "video_timestamp_s", "timestamp_difference_s", "note",
        ]
    ].sort_values(["family", "occurrence"])

    driving_starts = log_events[log_events["kind"] == "driving_start"][
        ["occurrence", "log_time_s"]
    ].rename(columns={"log_time_s": "log_driving_start_s"})
    driving_ends = log_events[log_events["kind"] == "driving_end"][
        ["occurrence", "log_time_s"]
    ].rename(columns={"log_time_s": "log_driving_end_s"})
    video_starts = observations[observations["family"] == "fts_driving_start"][
        ["occurrence", "video_timestamp_s"]
    ].rename(columns={"video_timestamp_s": "video_driving_start_s"})
    video_ends = observations[observations["family"] == "fts_driving_end"][
        ["occurrence", "video_timestamp_s"]
    ].rename(columns={"video_timestamp_s": "video_driving_end_s"})

    driving = driving_starts.merge(driving_ends, on="occurrence", validate="one_to_one")
    driving = driving.merge(video_starts, on="occurrence", how="left")
    driving = driving.merge(video_ends, on="occurrence", how="left")
    driving["log_driving_duration_s"] = (
        driving["log_driving_end_s"] - driving["log_driving_start_s"]
    )
    driving["video_driving_duration_s"] = (
        driving["video_driving_end_s"] - driving["video_driving_start_s"]
    )
    driving["driving_duration_difference_s"] = (
        driving["video_driving_duration_s"] - driving["log_driving_duration_s"]
    )

    return result, driving


comparison, driving_comparison = compare_with_video(visible_events, video_observations)
comparison

,family,occurrence,label,kind,log_timestamp,log_time_s,video_timestamp_s,timestamp_difference_s,note
